# AdapTrap - Four-Checkpoint Continual Learning (M1-M4) (4 Batches)

Extends the continual-learning design to 4 batches, under the identical structure as
M2 and M3:

- Contamination is tuned **once**, on the static model's (Batch 1) own validation set, and reused
  unchanged for every subsequent checkpoint -- not independently re-tuned per checkpoint.
- The `StandardScaler` is fit on all 4 batches' combined feature space.
- Full 4x4 evaluation matrix (16 conditions): every checkpoint
  evaluated against every batch's holdout.
- BWT and FWT use the standard multi-task formulas, averaged over all previously-seen /
  subsequently-introduced batches.

In [1]:
import numpy as np
import pandas as pd
from sklearn.ensemble import IsolationForest
from sklearn.metrics import silhouette_score

from adaptrap_pipeline import (
    process_batch, get_feature_cols, fit_scaler,
    assign_dominant_protocol, stratified_three_way_split, tune_contamination,
)

RANDOM_SEED = 42
np.random.seed(RANDOM_SEED)

BATCH_PATHS = {
    1: "CICHoneynet_July1.csv",
    2: "CICHoneynet_July2-9.csv",
    3: "CICHoneynet_July10-16.csv",
    4: "CICHoneynet_July17-23.csv"
}
N_BATCHES = len(BATCH_PATHS)


c:\Users\Lazaro\anaconda3\Lib\site-packages\pandas\core\arrays\masked.py:61: UserWarning: Pandas requires version '1.3.6' or newer of 'bottleneck' (version '1.3.5' currently installed).
  from pandas.core import (


## Load and profile all 4 batches

In [2]:
profiles = {}
for k, path in BATCH_PATHS.items():
    p = process_batch(path)
    p = assign_dominant_protocol(p)
    profiles[k] = p
    print(f"Batch {k} attacker profiles: {p.shape}")


c:\Users\Lazaro\Desktop\Enzo School Docs\THESIS2 - updated\adaptrap_pipeline.py:116: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  profiles = frame.groupby("source_ip").apply(_build_ip_profile).reset_index()


Batch 1 attacker profiles: (1179, 25)


c:\Users\Lazaro\Desktop\Enzo School Docs\THESIS2 - updated\adaptrap_pipeline.py:116: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  profiles = frame.groupby("source_ip").apply(_build_ip_profile).reset_index()


Batch 2 attacker profiles: (4485, 25)


c:\Users\Lazaro\Desktop\Enzo School Docs\THESIS2 - updated\adaptrap_pipeline.py:116: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  profiles = frame.groupby("source_ip").apply(_build_ip_profile).reset_index()


Batch 3 attacker profiles: (5474, 25)
Batch 4 attacker profiles: (5173, 25)


c:\Users\Lazaro\Desktop\Enzo School Docs\THESIS2 - updated\adaptrap_pipeline.py:116: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  profiles = frame.groupby("source_ip").apply(_build_ip_profile).reset_index()


## Align feature columns and fit one shared scaler across all 4 batches

In [3]:
feature_cols = sorted(set().union(*[get_feature_cols(profiles[k]) for k in profiles]))
for k in profiles:
    for col in feature_cols:
        if col not in profiles[k].columns:
            profiles[k][col] = 0

combined_for_scaling = pd.concat([profiles[k][feature_cols] for k in profiles])
scaler = fit_scaler(combined_for_scaling.assign(**{"__dummy__": 0}).drop(columns="__dummy__"), feature_cols)

def scale(p):
    return scaler.transform(p[feature_cols])


## Stratified 70/15/15 split per batch

In [4]:
split_idx = {}
X_train, X_val, X_test = {}, {}, {}

for k in profiles:
    train_idx, val_idx, test_idx = stratified_three_way_split(
        profiles[k], stratify_col="dominant_protocol", random_state=RANDOM_SEED
    )
    split_idx[k] = {"train": train_idx, "val": val_idx, "test": test_idx}
    X_train[k] = scale(profiles[k].iloc[train_idx])
    X_val[k]   = scale(profiles[k].iloc[val_idx])
    X_test[k]  = scale(profiles[k].iloc[test_idx])
    print(f"Batch {k} train/val/holdout: {len(X_train[k])}/{len(X_val[k])}/{len(X_test[k])}")


Batch 1 train/val/holdout: 825/177/177
Batch 2 train/val/holdout: 3139/673/673
Batch 3 train/val/holdout: 3831/821/822
Batch 4 train/val/holdout: 3621/776/776


## Contamination - tuned once, on Batch 1's validation set

Reused unchanged for every subsequent checkpoint.

In [5]:
contamination, m_static, contamination_results = tune_contamination(
    X_train[1], X_val[1], random_state=RANDOM_SEED
)
print(f"Selected contamination (reused for all checkpoints): {contamination}")
print(contamination_results)


Selected contamination (reused for all checkpoints): 0.0125
    contamination  flagging_rate  silhouette
0          0.0100       0.000000         NaN
1          0.0125       0.005650    0.718353
2          0.0150       0.005650    0.718353
3          0.0175       0.005650    0.718353
4          0.0200       0.011299    0.611903
5          0.0500       0.045198    0.469712
6          0.0750       0.056497    0.457703
7          0.1000       0.073446    0.444379
8          0.1250       0.084746    0.435791
9          0.1500       0.129944    0.369182
10         0.2000       0.214689    0.283334


## Train M1-M4, cumulative training data, SAME contamination throughout

M_k is trained on the cumulative combination of Batches 1..k.

In [6]:
models = {1: m_static}
for k in range(2, N_BATCHES + 1):
    X_cumulative_train = np.vstack([X_train[b] for b in range(1, k + 1)])
    m = IsolationForest(
        n_estimators=100, contamination=contamination,
        max_samples="auto", random_state=RANDOM_SEED,
    )
    m.fit(X_cumulative_train)
    models[k] = m
    print(f"M{k} trained on Batches 1-{k} ({len(X_cumulative_train)} profiles)")


M2 trained on Batches 1-2 (3964 profiles)
M3 trained on Batches 1-3 (7795 profiles)
M4 trained on Batches 1-4 (11416 profiles)


## Full evaluation matrix: every checkpoint (M1-M4) against every holdout (Batch 1-4)

R(i,j) = checkpoint i's model evaluated on batch j's holdout -- 16 conditions total.

In [7]:
def get_scores_flags(model, X):
    flags = (model.predict(X) == -1).astype(int)
    anomaly_scores = -model.decision_function(X)
    return anomaly_scores, flags

def evaluate(model, X_holdout):
    scores, flags = get_scores_flags(model, X_holdout)
    sil = silhouette_score(X_holdout, flags) if len(set(flags)) > 1 else np.nan
    return {"flagging_rate": flags.mean(), "score_mean": scores.mean(), "silhouette": sil}

R = {}
for i in range(1, N_BATCHES + 1):
    for j in range(1, N_BATCHES + 1):
        R[(i, j)] = evaluate(models[i], X_test[j])

results_df = pd.DataFrame([
    {"checkpoint": i, "evaluated_on_batch": j, **R[(i, j)]}
    for i in range(1, N_BATCHES + 1) for j in range(1, N_BATCHES + 1)
]).sort_values(["checkpoint", "evaluated_on_batch"])

print(results_df.to_string(index=False))


 checkpoint  evaluated_on_batch  flagging_rate  score_mean  silhouette
          1                   1       0.005650   -0.264766    0.707025
          1                   2       0.007429   -0.262204    0.694812
          1                   3       0.009732   -0.266906    0.773805
          1                   4       0.005155   -0.273161    0.761844
          2                   1       0.005650   -0.205217    0.707025
          2                   2       0.011887   -0.206649    0.672420
          2                   3       0.013382   -0.211268    0.744735
          2                   4       0.005155   -0.217395    0.761844
          3                   1       0.005650   -0.205149    0.707025
          3                   2       0.007429   -0.208920    0.694812
          3                   3       0.012165   -0.213776    0.755163
          3                   4       0.005155   -0.220040    0.761844
          4                   1       0.005650   -0.197654    0.707025
      

## BWT and FWT, standard multi-task formulas

- **BWT after checkpoint T** = average, over every previously-seen batch i < T, of
  `R(T,i) - R(i,i)`.
- **FWT** = average, over every batch i from 2 to N, of `R(i-1,i)`.

In [8]:
sil = lambda i, j: R[(i, j)]["silhouette"]

print("Per-checkpoint BWT (averaged over all previously-seen batches):")
bwt_per_checkpoint = {}
for T in range(2, N_BATCHES + 1):
    bwt_T = np.mean([sil(T, i) - sil(i, i) for i in range(1, T)])
    bwt_per_checkpoint[T] = bwt_T
    print(f"  BWT after checkpoint {T}: {bwt_T:+.4f}")

FWT = np.mean([sil(i - 1, i) for i in range(2, N_BATCHES + 1)])
print(f"\nFWT (averaged over all {N_BATCHES - 1} newly-introduced batches): {FWT:+.4f}")

print("\nPer-batch trend across checkpoints (is the effect stable, worsening, or recovering?):")
for j in range(1, N_BATCHES + 1):
    trend = " -> ".join(f"R({i},{j})={sil(i, j):.4f}" for i in range(j, N_BATCHES + 1))
    print(f"  Batch {j} holdout: {trend}")


Per-checkpoint BWT (averaged over all previously-seen batches):
  BWT after checkpoint 2: +0.0000
  BWT after checkpoint 3: +0.0112
  BWT after checkpoint 4: -0.0229

FWT (averaged over all 3 newly-introduced batches): +0.7338

Per-batch trend across checkpoints (is the effect stable, worsening, or recovering?):
  Batch 1 holdout: R(1,1)=0.7070 -> R(2,1)=0.7070 -> R(3,1)=0.7070 -> R(4,1)=0.7070
  Batch 2 holdout: R(2,2)=0.6724 -> R(3,2)=0.6948 -> R(4,2)=0.6407
  Batch 3 holdout: R(3,3)=0.7552 -> R(4,3)=0.7182
  Batch 4 holdout: R(4,4)=0.7618


## Firewall rule generation -- most current model

Two-label design: ALLOW / ESCALATE_TO_ANALYST, using the shared `generate_rules()` and
`get_representative_ports()` from `adaptrap_pipeline.py`. No automatic BLOCK -- every escalated
record requires human analyst review.

In [9]:
from adaptrap_pipeline import get_representative_ports, generate_rules

latest_checkpoint = N_BATCHES
port_lookup_latest = get_representative_ports(BATCH_PATHS[latest_checkpoint])

rules_latest = generate_rules(
    models[latest_checkpoint], X_test[latest_checkpoint],
    profiles[latest_checkpoint].iloc[split_idx[latest_checkpoint]["test"]],
    port_lookup_latest,
    label=f"M{latest_checkpoint} (most current) on Batch {latest_checkpoint} holdout"
)



--- M4 (most current) on Batch 4 holdout ---
         source_ip    port  anomaly_score               action
0  162.212.152.159    22.0       0.119255  ESCALATE_TO_ANALYST
1    87.103.208.30  1433.0       0.079354  ESCALATE_TO_ANALYST
2       81.5.95.27  1433.0       0.075903  ESCALATE_TO_ANALYST
3     59.148.20.65  1433.0       0.059166  ESCALATE_TO_ANALYST
4   162.142.125.13  8443.0      -0.000397  ESCALATE_TO_ANALYST
5   167.248.133.37    80.0      -0.021549  ESCALATE_TO_ANALYST
6    170.64.145.18    22.0      -0.034155  ESCALATE_TO_ANALYST
7   198.235.24.184  8443.0      -0.038115  ESCALATE_TO_ANALYST
8  143.110.219.176   135.0      -0.050555  ESCALATE_TO_ANALYST
9   198.199.104.48     NaN      -0.052857  ESCALATE_TO_ANALYST

ESCALATE_TO_ANALYST: 233, ALLOW: 543
(No records are auto-blocked. BLOCK is an analyst decision made after reviewing an ESCALATE_TO_ANALYST record, not an automated label.)
